# Ant Colony Optimization per il Problema del Flusso Massimo
## Progetto di Heuristics & Metaheuristics For Optimization And Learning
### Giulio Pedicone (Matricola: 1000084718)

---

Questo progetto presenta un'implementazione dell'Ant Colony System (ACS) adattato per risolvere il Problema del Flusso Massimo. L'algoritmo utilizza formiche artificiali per trovare iterativamente percorsi aumentanti in una rete di flusso, con tracce di feromoni che guidano il processo di ricerca. L'implementazione include aggiornamenti locali e globali dei feromoni, regole di transizione sofisticate e criteri di terminazione adattivi specificamente progettati per problemi di ottimizzazione del flusso.

In [ ]:
import numpy as np
import statistics
import random
from FlowNetwork import FlowNetwork
import matplotlib.pyplot as plt

# Parametri dell'algoritmo

Modifica questi parametri per effettuare cambiamenti sul funzionamento di ACO

- α (pheromone_weight): Controlla l'influenza delle tracce di feromoni
- β (heuristic_weight): Controlla l'influenza dell'informazione euristica
- q₀: Parametro di sfruttamento vs esplorazione
- φ (local_evaporation_rate): Tasso di decadimento locale dei feromoni
- ρ (global_evaporation_rate): Tasso di decadimento globale dei feromoni
- τ₀ (initial_pheromone): Livello iniziale di feromoni

In [ ]:
PHEROMONE_WEIGHT = 1.0          # α - Controls pheromone importance
HEURISTIC_WEIGHT = 3.0          # β - Controls heuristic importance
Q0 = 0.8                        # q0 - Parameter controlling exploitation vs exploration
LOCAL_EVAPORATION_RATE = 0.1    # φ - Local pheromone evaporation rate
GLOBAL_EVAPORATION_RATE = 0.3   # ρ - Global pheromone evaporation rate
INITIAL_PHEROMONE = 0.01       # τ0 - Initial pheromone level

# Visualizzazione del grafico

In [ ]:
def plot_convergence_analysis(all_run_flows, theoretical_max_flow, experiment_statistics):
    plt.figure(figsize=(12, 8))
    color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                     '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

    final_flows = [flow_history[-1] if flow_history else 0 for flow_history in all_run_flows]
    maximum_flow_achieved = max(final_flows)

    optimal_runs = []
    for run_index, final_flow in enumerate(final_flows):
        if abs(final_flow - maximum_flow_achieved) < 1e-6:
            optimal_runs.append((run_index, len(all_run_flows[run_index])))

    best_run_index = min(optimal_runs, key=lambda x: x[1])[0]

    for run_index, flow_history in enumerate(all_run_flows):
        iteration_sequence = list(range(len(flow_history)))
        total_iterations = len(flow_history)
        is_best_run = (run_index == best_run_index)

        if is_best_run:
            line_width = 4
            line_color = '#0000ff'
            line_alpha = 1.0
            z_order = 10
        else:
            line_width = 2
            line_color = color_palette[run_index % len(color_palette)]
            line_alpha = 0.8
            z_order = 5

        run_label = f'Run {run_index + 1} ({total_iterations} iter)'
        if is_best_run:
            run_label += ' - BEST'

        plt.plot(iteration_sequence, flow_history, color=line_color, linewidth=line_width, 
                 label=run_label, alpha=line_alpha, zorder=z_order)

    plt.axhline(y=theoretical_max_flow, color='red', linestyle='--', 
                linewidth=2, label=f'Theoretical Max: {theoretical_max_flow:.0f}')

    plt.xlabel('Iteration')
    plt.ylabel('Total Flow')
    plt.title('ACS (Ant Colony System) Convergence Analysis - Maximum Flow Problem')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    statistics_text = (f'Best: {experiment_statistics["best_value"]:.0f}\n'
                       f'Mean: {experiment_statistics["mean_value"]:.1f}\n'
                       f'Std Dev: {experiment_statistics["standard_deviation"]:.1f}')

    plt.text(0.98, 0.02, statistics_text, transform=plt.gca().transAxes, 
             fontsize=11, verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

    plt.tight_layout()
    plt.show()

# Implementazione dell'algoritmo ACS

In [ ]:
def ant_colony_system(flow_network, max_iterations=20000, 
                      pheromone_weight=1.0, heuristic_weight=2.0, 
                      q0=0.9, local_evaporation_rate=0.1, 
                      global_evaporation_rate=0.1, initial_pheromone=1.0):
    """
    Implementazione dell'ACS adattato per il problema di Maximum Flow.
    """
    total_flow = 0.0                    # Flusso totale accumulato (obiettivo da massimizzare)
    global_best_flow = 0.0              # Miglior flusso globale trovato
    global_best_path = []               # Miglior path che ha contribuito al flusso
    flow_progression = []               # Storia della convergenza
    function_evaluations = 0            # Contatore valutazioni funzione obiettivo
    best_solution_iteration = 0         # Iterazione in cui è stato trovato il migliore
    best_solution_evaluations = 0       # Valutazioni per raggiungere il migliore

    for current_iteration in range(1, max_iterations + 1):
        # DIFFERENZA CON ACS CLASSICO: Numero di formiche adattivo
        ant_count = max(1, len(flow_network.get_successors(flow_network.source_node)))
        current_solutions = []

        for ant_id in range(ant_count):
            # DIFFERENZA CON ACS CLASSICO: Ogni formica parte sempre dalla source
            current_node = flow_network.source_node
            constructed_path = [current_node]
            edge_capacities = []                
            visited_nodes = set([current_node]) 

            # COSTRUZIONE DEL PATH
            while current_node != flow_network.sink_node:
                # Filtro per archi con capacità residua > 0

                feasible_neighbors = [
                    neighbor for neighbor in flow_network.get_successors(current_node)
                    if neighbor not in visited_nodes and 
                    flow_network.get_residual_capacity(current_node, neighbor) > 0
                ]

                if not feasible_neighbors:
                    break  

                # REGOLA DI TRANSIZIONE ACS: Simile al classico ma con soft exploitation
                next_node = flow_network.acs_state_transition_rule(
                    current_node, q0, pheromone_weight, heuristic_weight
                )

                if next_node is None:
                    break

                # AGGIORNAMENTO LOCALE
                flow_network.acs_local_pheromone_update(
                    current_node, next_node, local_evaporation_rate, initial_pheromone
                )

                # Qui raccogliamo capacità per calcolare il flusso del path
                residual_capacity = flow_network.get_residual_capacity(current_node, next_node)
                edge_capacities.append(residual_capacity)
                constructed_path.append(next_node)
                visited_nodes.add(next_node)
                current_node = next_node

            # VALUTAZIONE SOLUZIONE
            if current_node == flow_network.sink_node and edge_capacities:

                path_flow = min(edge_capacities)
                function_evaluations += 1

                solution = {
                    'path': constructed_path.copy(),
                    'flow_value': path_flow,
                    'is_valid': True
                }
                current_solutions.append(solution)


                flow_network.update_path_capacities(constructed_path, path_flow)
                total_flow += path_flow
            else:
                # Path non valido (non raggiunge il sink)
                current_solutions.append({
                    'path': constructed_path.copy(),
                    'flow_value': 0.0,
                    'is_valid': False
                })

        # AGGIORNAMENTO MIGLIOR SOLUZIONE
        valid_solutions = [s for s in current_solutions if s['is_valid']]

        if valid_solutions and total_flow > global_best_flow:
            global_best_flow = total_flow
            best_solution_iteration = current_iteration
            best_solution_evaluations = function_evaluations
            # Selezione del path con maggior contributo al flusso
            iteration_best_solution = max(valid_solutions, key=lambda x: x['flow_value'])
            global_best_path = iteration_best_solution['path']

        flow_progression.append(total_flow)

        # AGGIORNAMENTO GLOBALE
        if global_best_path and global_best_flow > 0:
            flow_network.acs_global_pheromone_update(
                global_best_path, global_best_flow, global_evaporation_rate
            )

        # CRITERI DI TERMINAZIONE: Specifici per il maximum flow
        # 1. Raggiungimento del flusso teorico massimo
        if abs(total_flow - flow_network.theoretical_max_flow) < 1e-6:
            break
        # 2. Nessun path augmentante trovato
        if not valid_solutions:
            break

    return (total_flow, flow_progression, best_solution_iteration, best_solution_evaluations)


# Esecuzione dell'algoritmo

Parametri di Test

- Numero di esecuzioni: 10 esecuzioni indipendenti
- Iterazioni massime: 20.000 per esecuzione
- Istanza di test: Selezionabile nella cartella istanze
- Semi casuali: Seme diverso per ogni esecuzione per garantire validità statistica

In [ ]:
def run_experimental_analysis(original_network, experiment_runs=10, max_iterations=20000):
    print(f"Starting ACS experimental analysis with {experiment_runs} independent runs")
    print("ACS Parameters:")
    print(f"  α (pheromone weight): {PHEROMONE_WEIGHT}")
    print(f"  β (heuristic weight): {HEURISTIC_WEIGHT}")
    print(f"  q0 (exploitation parameter): {Q0}")
    print(f"  φ (local evaporation): {LOCAL_EVAPORATION_RATE}")
    print(f"  ρ (global evaporation): {GLOBAL_EVAPORATION_RATE}")
    print(f"  τ0 (initial pheromone): {INITIAL_PHEROMONE}")
    print("=" * 70)

    flow_results = []
    iteration_results = []
    evaluation_results = []
    convergence_histories = []
    random_seeds = random.sample(range(1_000_000), experiment_runs)

    for run_id in range(experiment_runs):
        print(f"Executing ACS run {run_id + 1}/{experiment_runs} (seed: {random_seeds[run_id]})")
        np.random.seed(random_seeds[run_id])
        random.seed(random_seeds[run_id])

        current_network = original_network.copy()
        current_network.reset_capacities()

        final_flow, flow_history, best_iteration, best_evaluations = ant_colony_system(
            current_network, max_iterations, PHEROMONE_WEIGHT, HEURISTIC_WEIGHT, 
            Q0, LOCAL_EVAPORATION_RATE, GLOBAL_EVAPORATION_RATE, INITIAL_PHEROMONE
        )

        flow_results.append(final_flow)
        iteration_results.append(best_iteration)
        evaluation_results.append(best_evaluations)
        convergence_histories.append(flow_history)

        print(f"  Completed: Flow={final_flow:.2f}, Iterations={len(flow_history)}")

    best_value = max(flow_results)
    mean_value = statistics.mean(flow_results)
    standard_deviation = statistics.stdev(flow_results) if len(flow_results) > 1 else 0
    average_iterations = statistics.mean(iteration_results)
    average_evaluations = statistics.mean(evaluation_results)

    final_statistics = {
        'best_value': best_value,
        'mean_value': mean_value,
        'standard_deviation': standard_deviation,
        'average_iterations_to_best': average_iterations,
        'average_evaluations_to_best': average_evaluations
    }

    print("\n" + "=" * 70)
    print("FINAL REPORT - ACS EXPERIMENTAL ANALYSIS")
    print("=" * 70)
    print(f"Maximum flow value achieved: {best_value:.2f}")
    print(f"Average maximum flow: {mean_value:.2f}")
    print(f"Standard deviation: {standard_deviation:.2f}")
    print(f"Average iterations to reach best: {average_iterations:.1f}")
    print(f"Average evaluations to reach best: {average_evaluations:.1f}")
    print(f"Success rate (optimal solutions): {sum(1 for f in flow_results if abs(f - original_network.theoretical_max_flow) < 1e-6) / len(flow_results) * 100:.1f}%")
    print("=" * 70)

    print("Generating ACS convergence analysis plot...")
    plot_convergence_analysis(convergence_histories, 
                              original_network.theoretical_max_flow, 
                              final_statistics)

    return final_statistics


# Main

In [ ]:
if __name__ == "__main__":
    print("Initializing ACS for Maximum Flow Problem...")
    original_network = FlowNetwork("istanze/network_160.txt")
    original_network.get_network_info()

    print("\n" + "=" * 70)
    print("STARTING ACS (ANT COLONY SYSTEM) ANALYSIS")
    print("=" * 70)

    experiment_results = run_experimental_analysis(
        original_network, experiment_runs=10, max_iterations=20000
    )